In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# ====================== Model Training ======================
FILE_PATH = "model_training.xlsx"
SAVE_MODEL_DIR = "models/qing_citywall_model"

df = pd.read_excel(FILE_PATH)
texts = df["Abstract"].astype(str).tolist()
labels = df["Relation"].tolist()

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.1, random_state=42
)

tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")
model = BertForSequenceClassification.from_pretrained("bert-base-chinese", num_labels=2)

def encode(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

train_encodings = encode(train_texts)
test_encodings = encode(test_texts)

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
test_dataset = Dataset(test_encodings, test_labels)

training_args = TrainingArguments(
    output_dir=SAVE_MODEL_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

model.save_pretrained(SAVE_MODEL_DIR)
tokenizer.save_pretrained(SAVE_MODEL_DIR)
print("Training complete! Model saved at:", SAVE_MODEL_DIR)

In [ ]:
MODEL_DIR = "models/qing_citywall_model"
UNLABELED_FILE = "correlation_demo_input.xlsx"
OUTPUT_FILE = "correlation_demo_output.xlsx"


model = BertForSequenceClassification.from_pretrained(MODEL_DIR)
tokenizer = BertTokenizer.from_pretrained(MODEL_DIR)
model.eval()

df = pd.read_excel(UNLABELED_FILE)
texts = df["Abstract"].astype(str).tolist()

def predict(text):
    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    label = torch.argmax(probs).item()
    confidence = round(float(probs[0][label]), 4)
    return label, confidence

results = [predict(t) for t in texts]

df["pred_relation"] = [r[0] for r in results]
df["confidence"] = [r[1] for r in results]
df.to_excel(OUTPUT_FILE, index=False)

print(f"Prediction complete! Results saved at:{OUTPUT_FILE}")